# KIKA: Serpent Sensitivity Analysis Report Pipeline

This notebook demonstrates the complete pipeline for sensitivity analysis using Serpent adjoint-weighted calculations:

1. **Parse Serpent .sens file** → `SensitivityFile`
2. **Generate comprehensive report** → HTML with sensitivity plots
3. **Export to SDF format** → For comparison and downstream UQ
4. **Run uncertainty quantification** → Sandwich formula with covariance matrices

The new `SerpentSensReport` class provides a standardized way to analyze Serpent sensitivity results.

# KIKA: Serpent Sensitivity Analysis Report Pipeline

This notebook demonstrates the complete pipeline for sensitivity analysis and uncertainty quantification using Serpent adjoint-weighted sensitivity calculations:

1. **Parse Serpent .sens file** → `SensitivityFile`
2. **Generate comprehensive report** → HTML with sensitivity plots
3. **Export to SDF format** → For comparison and downstream UQ
4. **Run uncertainty quantification** → Sandwich formula with covariance matrices

## Architecture Overview

```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│   Serpent       │     │ SensitivityFile  │     │    SDFData      │
│   .sens File    │ ──► │   (Full data)    │ ──► │ (Common format) │
└─────────────────┘     └──────────────────┘     └─────────────────┘
                               │                        │
                               ▼                        ▼
                        ┌────────────────┐       ┌──────────────┐
                        │SerpentSensReport│       │  UQReport    │
                        │  (Full report)  │       │ (Sandwich)   │
                        └────────────────┘       └──────────────┘
```

## Key Differences from MCNP

| Aspect | MCNP (PERT) | Serpent |
|--------|-------------|----------|
| Method | Differential operator | Adjoint-weighted perturbation |
| Second-order | Available (method=3) | Not available |
| Materials | Single nuclide per run | Multiple materials/nuclides |
| Legendre | Not native | Native (MT 4001+) |
| Output | MCTAL + input parsing | Direct .sens file |

In [ ]:
import kika
from kika.serpent.parse_sens import read_sensitivity_file
from kika.sensitivities import (
    SerpentSensReport,
    UQReport,
    ComparisonReport
)
from kika.sensitivities.sensitivity_processing import create_sdf_from_serpent
from pathlib import Path
import pandas as pd

# Setup paths
repo_root = Path.cwd().resolve().parent
output_dir = Path.cwd() / 'output'
output_dir.mkdir(exist_ok=True)

print(f"Output directory: {output_dir}")

## Step 1: Parse Serpent Sensitivity File

The `read_sensitivity_file` function parses Serpent .sens (or _sens0.m) output files and creates a `SensitivityFile` object containing:
- Material definitions
- Nuclide ZAI values
- Perturbation types (MT reactions, Legendre moments)
- Energy-dependent and integrated sensitivity values

In [ ]:
# Path to Serpent sensitivity file
# Replace with your actual Serpent .sens file path
serpent_sens_file = repo_root / 'kika' / 'serpent' / 'PWRSphere.sss2_sens0.m'

# Check if file exists
if not serpent_sens_file.exists():
    print(f"WARNING: Example file not found at {serpent_sens_file}")
    print("Please update the path to point to your Serpent .sens file.")
    serpent_sens_file = None
else:
    print(f"Loading: {serpent_sens_file}")

In [ ]:
if serpent_sens_file:
    # Parse the Serpent sensitivity file
    sf = read_sensitivity_file(str(serpent_sens_file))
    
    # Display summary
    print(sf.summary())
else:
    print("Skipping - no Serpent file available")
    sf = None

In [ ]:
if sf:
    # Explore the data structure
    print(f"Number of materials: {sf.n_materials}")
    print(f"Number of nuclides: {sf.n_nuclides}")
    print(f"Number of perturbations: {sf.n_perturbations}")
    print(f"Energy bins: {sf.n_energy_bins}")
    print(f"\nAvailable responses: {sf.responses}")
    print(f"\nMT reactions: {sf.reactions}")
    
    # Show materials
    print("\nMaterials:")
    for mat in sf.materials:
        print(f"  {mat.index}: {mat.name}")
    
    # Show nuclides (first 5)
    print("\nNuclides (first 5):")
    for nuc in sf.nuclides[:5]:
        print(f"  {nuc.index}: ZAI {nuc.zai}")

## Step 2: Generate Serpent Sensitivity Report

The `SerpentSensReport` class generates comprehensive reports from Serpent sensitivity data:
- Summary tables
- Integrated sensitivity tables
- Energy-dependent sensitivity plots
- Legendre moment analysis

In [ ]:
if sf:
    # Get the first response for analysis
    response = sf.responses[0]
    print(f"Analyzing response: {response}")
    
    # Create the report
    report = SerpentSensReport(
        sensitivity_file=sf,
        response=response,
        title="Serpent Sensitivity Analysis - Demo"
    )
    
    # View summary
    report.summary()
else:
    report = None
    print("Skipping - no Serpent data")

In [ ]:
if report:
    # View integrated sensitivity table
    sens_table = report.sensitivity_table(response=response)
    print(f"Found {len(sens_table)} reaction sensitivities")
    sens_table.head(15)

In [ ]:
if report:
    # Plot top integrated sensitivities
    fig = report.plot_sensitivities(response=response, top_n=10)
    fig

### Energy-Dependent Sensitivity Plots

Serpent provides energy-dependent sensitivities. Let's visualize them.

In [ ]:
if sf:
    # Use the built-in plotting from SensitivityFile
    # This shows sensitivity per lethargy vs energy
    
    # Plot for a specific material, nuclide, and reactions
    sf.plot(
        response=response,
        mat=0,  # First material
        zai=0,  # First nuclide
        mt=[2, 18, 102],  # Elastic, fission, capture
        per_lethargy=True,
        errorbars=True,
        energy_range=(1e-3, 20)  # 1 meV to 20 MeV
    )

### Legendre Moment Sensitivities

Serpent can calculate sensitivities to Legendre moments of angular distributions (elastic scattering). These are stored with MT numbers 4001, 4002, etc. (corresponding to L=1, L=2, ...).

In [ ]:
if sf:
    # Check if Legendre sensitivities are available
    legendre_mts = [mt for mt in sf.reactions if mt >= 4000]
    if legendre_mts:
        print(f"Legendre moments available: MT {legendre_mts}")
        print("(MT 4001 = L=1, MT 4002 = L=2, etc.)")
        
        # Plot Legendre sensitivities
        sf.plot(
            response=response,
            mat=0,
            zai=0,
            leg=[1, 2, 3],  # L=1, L=2, L=3 moments
            per_lethargy=True,
            energy_range=(0.1, 10)
        )
    else:
        print("No Legendre moment sensitivities in this file")

### Save HTML Report

In [ ]:
if report:
    # Save HTML report
    report_path = output_dir / 'serpent_sens_report.html'
    report.save_html(
        filepath=str(report_path),
        response=response,
        include_plots=True
    )
    print(f"Report saved: {report_path}")

## Step 3: Export to SDF Format

Convert Serpent sensitivities to SDF format for:
- Comparison with MCNP/SCALE results
- Uncertainty quantification using the sandwich formula

In [ ]:
if sf:
    # Create SDF from Serpent data
    sdf_data = create_sdf_from_serpent(
        serpent_file=sf,
        response_name=response,
        title='Serpent_Demo',
        # Optional filters:
        # material_filter=['fuel'],
        # nuclide_filter=[92235, 92238],
        # mt_filter=[2, 18, 102],
        response_values=(1.0, 0.01)  # (r0, e0) - response value and relative error
    )
    
    # Display SDF structure
    sdf_data
else:
    sdf_data = None
    print("Skipping - no Serpent data")

In [ ]:
if sdf_data:
    # Save SDF file
    sdf_data.write_file(output_dir=str(output_dir))
    
    # Show first few reactions in SDF
    print(f"\nSDF contains {len(sdf_data.data)} reaction profiles")
    for i, r in enumerate(sdf_data.data[:5]):
        int_sens = sum(r.sensitivity)
        print(f"  {r.nuclide} {r.reaction_name}: {int_sens:.4e}")

## Step 4: Uncertainty Quantification

Use the sandwich formula with covariance matrices to propagate uncertainties.

In [ ]:
# UQ requires covariance matrices
# Here's the complete workflow:

# from kika.cov import CovMat
# from kika.cov.multigroup import MGMF34CovMat  # For Legendre moments
# 
# # Load covariance matrices
# cov_u235 = CovMat.from_file('path/to/u235_covmat.dat')
# cov_u238 = CovMat.from_file('path/to/u238_covmat.dat')
# 
# # For Legendre moments (if available)
# leg_cov = MGMF34CovMat.from_file('path/to/mf34_covmat.dat')
# 
# # Create UQ report
# uq_report = UQReport(
#     sdf_data=sdf_data,
#     cov_mat=[cov_u235, cov_u238],  # Can provide multiple
#     legendre_cov_mat=leg_cov,       # Optional for Legendre
#     title='Serpent Uncertainty Quantification'
# )
# 
# # Run and view results
# uq_report.compute()
# print(uq_report.result)

print("UQ workflow requires covariance matrices.")
print("The sandwich formula supports both cross-section and Legendre sensitivities.")

## Working with Multiple Serpent Files

The `create_sdf_from_serpent` function can combine sensitivities from multiple Serpent files (e.g., from different detector responses or materials).

In [ ]:
# Example of combining multiple files
#
# sf1 = read_sensitivity_file('case1_sens0.m')
# sf2 = read_sensitivity_file('case2_sens0.m')
#
# sdf = create_sdf_from_serpent(
#     serpent_file=[sf1, sf2],
#     response_name=['sens_ratio_BIN_1', 'sens_ratio_BIN_2'],
#     title='Combined_Sensitivities'
# )

print("Multiple files can be combined with matching response names")

## Summary

This notebook demonstrated the Serpent sensitivity analysis pipeline:

| Step | Function/Class | Output |
|------|---------------|--------|
| 1. Parse Serpent | `read_sensitivity_file()` | `SensitivityFile` |
| 2. Generate report | `SerpentSensReport` | HTML with plots |
| 3. Export to SDF | `create_sdf_from_serpent()` | `SDFData` → .sdf file |
| 4. Run UQ | `UQReport` | Uncertainty breakdown |

### Key Serpent Features

- **Multi-material, multi-nuclide** in single file
- **Legendre moment sensitivities** (MT 4001+ for L=1+)
- **Energy-dependent data** with lethargy normalization
- **Direct adjoint-weighted** perturbation (no second-order)

### Comparison with MCNP

Both Serpent and MCNP sensitivities can be converted to SDFData for:
- Cross-code comparison (see `ComparisonReport`)
- Unified UQ pipeline (see `UQReport`)

In [ ]:
# List generated outputs
print("Generated outputs:")
for f in output_dir.glob('*'):
    print(f"  - {f.name}")